# Studying nonconvex constellations via $\Delta\Phi(x,p)$

We apply $\Delta\Phi(x,p)$ and its visualizations to the study of a couple of Engelsma's 
examples of nonconvex constellations.  These are relatively long admissible constellations of low span
such that
$$ \pi(|s|) < {\rm length}(s).$$
These examples $s$ show that if the $k$-tuple conjecture is true, then the convexity conjecture
for $\pi(x)$ is false.  For such an $s$ we have
$$ \pi(\gamma_0+|s|) > \pi(\gamma_0) + \pi(|s|).$$

We analyze a counterexample $s$ with $(J,|s|)=(459,3242)$.
We start with a driving term for $s$ in ${\mathcal G}(11^\#)$ with initial generator $\gamma_0 = 1271$ and trace its evolution through subsequent stages of the sieve.

Guided by step R2 of the recursion $R: {\mathcal G}(p_{k-1}^\#) \longrightarrow {\mathcal G}(p_k^\#)$,
we use primorial coordinates or the primorial expansion for $\gamma_0(p_k)$ to track the incidences of $s$.
$$ \gamma_0(p_k) = \gamma_0 + m_1 \cdot 11^\# + m_2 \cdot 13^\# + \cdots + m_k \cdot p_{k-1}^\#$$
or 
$$ \gamma_0(p_k) = \gamma_0 + 11^\# (m_1 + 13(m_2 + 17(m_3 + \cdots +  p_{k-1}^\# \cdot m_k ))\cdots)$$
Each coefficient $m_i$ lies in the range $0 \le m_i < p_i$ and indicates which copy of ${\mathcal G}(p_{i-1}^\#)$ this image of $s$ lies in
under step R2 for ${\mathcal G}(p_{i-1}^\#)\longrightarrow {\mathcal G}(p_i^\#)$.

In [1]:
import pandas as pd
import numpy as np
import array
import itertools
from sympy import mod_inverse
import random

import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
plt.rcParams['figure.dpi'] = 300
plt.ion

import gc
import psutil
import sys
import csv
import pickle

## Engelsma counterexamples to the convexity conjecture
Engelsma et al. have identified several examples of admissible constellations for which their lengths $J$ exceed the number of primes under their span
$$ \pi(|s|) < J$$
The shortest counterexamples $s$ have length $J=458$ and span $|s|=3240$.  These counterexamples can be extended by a single gap $2$ to produce a second admissible counterexample of length $J=459$ and span $|s|=3242$.

There are $58$ distinct $(459,3242)$-counterexamples.  These constellations all start from two driving terms
in ${\mathcal G}(11^\#)$, one starting at $\gamma_0=107$ and the other (it's mirror image) starting at $\gamma_0=1271$.
Of the $58$ constellations, half have the same driving term $\tilde{s}$ up into ${\mathcal G}(59^\#)$.  The other half share
a single driving term that is the reflection of $\tilde{s}$.


In [2]:
# set up the array of small primes.  Start with primes19
smallp=np.load('primesE9.npy')
smallp = np.concatenate(([2],smallp))

In [3]:
# PRIMES:  The array smallp runs through primes from 2 up to 1023101273
# 
smallp[0:52], smallp[250:270],smallp[450:470],smallp[-5:]

(array([  2,   3,   5,   7,  11,  13,  17,  19,  23,  29,  31,  37,  41,
         43,  47,  53,  59,  61,  67,  71,  73,  79,  83,  89,  97, 101,
        103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167,
        173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239]),
 array([1597, 1601, 1607, 1609, 1613, 1619, 1621, 1627, 1637, 1657, 1663,
        1667, 1669, 1693, 1697, 1699, 1709, 1721, 1723, 1733]),
 array([3187, 3191, 3203, 3209, 3217, 3221, 3229, 3251, 3253, 3257, 3259,
        3271, 3299, 3301, 3307, 3313, 3319, 3323, 3329, 3331]),
 array([1023101207, 1023101221, 1023101263, 1023101269, 1023101273]))

## Greedy and opportunistic searches for surviving instances
This notebook takes the results of the breadth-first search of '23_nonconvex_prep_Engelsma459' and
conducts greedy and opportunistic depth-first searchs for a surviving counterexample.

For an instance to survive the sieve, its primorial coordinates have to have a long sequence of $0$'s.
That is, the instance occurs in the first copy of ${\mathcal G}(p^\#)$ under step R2 and is not eliminated until
$\gamma_0$ itself is confirmed as a prime.

In [4]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 3196.28 MB


In [5]:
# This function returns a list of available residues mod inp that begin admissible images of constellation incons
def admissible(inp, incons):
    # calculate list of covered residues mod p by the input constellation
    rez=0
    covered_rez={0}
    i=0
    while (i < len(incons)):
        rez = (rez + incons[i])% inp
        if rez not in covered_rez:
            covered_rez.add(rez)
        i += 1
    # are all residues covered?
    n_available_rez = inp - len(covered_rez)
    i=1
    available_rez = set()
    # each entry in covered_rez corresponds to a starting residue of (inp-rez)mod inp
    while (i < inp):
        test_rez = inp - i
        if test_rez not in covered_rez:
            available_rez.add(i)
        i += 1
    return available_rez

In [6]:
# this function calculates the value of mk such that 
#  0 <= mk < pk  and  mk*pml(p_{k-1}) + r0 = rk mod pk
# reminder for indexing that pk = smallp[k+4], a shift of 4, and p0=11
def primorialm(k,rk,r0):
    pk = smallp[k+4]
    i = 0
    pmlp = 1
    while (i < k+4):
        pmlp = (pmlp * smallp[i]) % pk
        i += 1
    mk = (mod_inverse(pmlp, pk) * (rk-r0) ) % pk
    return mk

In [7]:
# This function returns the generator in G(11#) that begins a driving term for the input constellation
def findgamma11(incons):
    i=1
    gamma0 = 1
    pmlp = 2
    
    while (i <= 4):  # smallp[4]=11
        pk = int(smallp[i])
        rez = admissible(pk, incons)

        if (len(rez) != 1):   # We assume that the generator in G(11#) is unique
            print(f"UNEXPECTED: p {pk} rez {len(rez)} {rez}")

        target_r = int(list(rez)[0])
        r0 = gamma0 % pk
        mk = (mod_inverse(pmlp, pk) * (target_r-r0)) % pk

        gamma0 += mk*pmlp
        pmlp = (pmlp * smallp[i])

        i += 1

    return gamma0        


## Main loop through Engelsma (459,3242)-counterexamples
What follows is the main loop through the 58 counterexamples identified by Thomas Engelsma for the case $(J,|s|)=(459,3242)$.

We start with the primorial expansions that end in one or more $m_k=0$ and pursue depth-first searches for long
sequences of $m_k=0$.

The indices in the data files correspond to the constellations in the order given by 'Eng459_sorted' and 'Eng459_prefixes'


In [8]:
# XXXQHERE [6/27] - 

In [9]:
# reading in the (459,3242) constellations
Eng459_sortedB = []
with open('Eng459_sorted.csv', 'r') as fqtr:
    ssort_reader = csv.reader(fqtr)
    for row in ssort_reader:
        Eng459_sortedB.append([int(x) for x in row])
fqtr.close()

In [10]:
# reading in the unique prefix for each of the (459,3242) constellations
Eng459_prefixesB = []
with open('Eng459_prefixes.csv', 'r') as fqtr:
    pref_reader = csv.reader(fqtr)
    for row in pref_reader:
        Eng459_prefixesB.append([int(x) for x in row])
fqtr.close()

In [12]:
# average gap size for counterexample and for primes
3242/459, 3253/459

(7.06318082788671, 7.087145969498911)

In [21]:
# parameters for the random search
probelength = 8000
Engdex = 32   # INDEX of (459,3242)-counterexample for following loop
maxruns = np.zeros(58, dtype=int)
maxdexes = np.zeros(58, dtype=int)

In [22]:
# LOOP on Engdex from 0 to 57

In [23]:
# The next few cells parse the "Eng459_xx_results.csv" files
# 1. Extract the data into a list of strings, by row
filenameresults = 'Eng459_'+str(Engdex)+'_results.csv'
Eng459_str_data = []
with open(filenameresults, 'r') as fqtr:
    data_reader = csv.reader(fqtr)
    for row in data_reader:
        Eng459_str_data.append([x for x in row])
fqtr.close()

In [24]:
Eng459_str_data

[['32',
  '28',
  '137',
  '14',
  '211',
  '38016000',
  '[   2    2    1    1    3    2    2    2    5    4   10    6    5    6\n   11   10   12    8   13    9   17   15   17   27   21   17   21   28\n   22   31   33   38   35   38   41   49   54   49   55   60   64   71\n   66   74   83   89   83   89   94   98  106  100  101  107  116  112\n  114  128  127  135  130  147  140  149  148  159  151  167  179  180\n  187  196  196  203  201  218  220  216  223  227  229  240  248  250\n  257  255  263  265  269  282  277  284  296  298  307  318  334  331\n  332  340  344  351  351  363  358  378  383  395  399  401  406  409\n  410  416  437  436  436  439  463  465  471  463  488  488  498  502\n  518  514  519  528  548  541  546  554  564  571  581  586  587  597\n  598  608  606  616  627  631  635  635  658  656  666  659  670  680\n  685  694  696  720  717  727  737  747  747  762  764  775  779  790\n  786  794  802  808  818  839  839  845  849  847  859  861  861  865\n  877

In [25]:
# 2. Assign the first numeric results to variables
aic = int(Eng459_str_data[0][0])       # index of this constellation in Eng459_sorted
apref_len = int(Eng459_str_data[0][1]) # length of the unique prefix for the primorial coordinates
ap0 = int(Eng459_str_data[0][2])       # the first prime for the admissible residues beyond the prefix
aextlen = int(Eng459_str_data[0][3])   # length of the exhaustive breadth-first extensions explored for this constellation
aextmaxp = int(Eng459_str_data[0][4])  # the maximum prime for the extensions explored
anumext = int(Eng459_str_data[0][5])   # the number of extensions explored in this breadth-first approach

In [26]:
Eng459_str_data[0][0:6], aic, apref_len, ap0,aextlen, aextmaxp, anumext

(['32', '28', '137', '14', '211', '38016000'], 32, 28, 137, 14, 211, 38016000)

In [27]:
# 3. extract the array of numbers of admissible residues beyond the prefix
str_numadm = (Eng459_str_data[0][6])[1:-1]    # throwing out the brackets [] at beginning and end
numadm = np.fromstring(str_numadm, dtype=int, sep=' ')


In [28]:
Eng459_sortedB[Engdex]

[2,
 4,
 2,
 4,
 8,
 6,
 4,
 2,
 10,
 6,
 2,
 6,
 12,
 4,
 6,
 12,
 2,
 4,
 2,
 4,
 8,
 6,
 12,
 4,
 6,
 8,
 6,
 4,
 2,
 4,
 14,
 10,
 12,
 2,
 10,
 2,
 4,
 12,
 2,
 10,
 2,
 4,
 6,
 8,
 6,
 6,
 6,
 4,
 6,
 12,
 6,
 2,
 4,
 8,
 6,
 10,
 2,
 4,
 8,
 16,
 6,
 6,
 2,
 6,
 10,
 2,
 22,
 2,
 6,
 4,
 6,
 2,
 10,
 12,
 8,
 6,
 6,
 6,
 4,
 6,
 8,
 4,
 2,
 4,
 2,
 18,
 10,
 2,
 10,
 14,
 4,
 14,
 10,
 2,
 4,
 12,
 2,
 18,
 10,
 2,
 6,
 6,
 10,
 18,
 2,
 10,
 6,
 8,
 6,
 4,
 14,
 6,
 10,
 6,
 6,
 8,
 6,
 10,
 6,
 8,
 4,
 2,
 6,
 10,
 12,
 6,
 12,
 2,
 6,
 4,
 2,
 4,
 6,
 6,
 2,
 10,
 12,
 14,
 4,
 8,
 12,
 6,
 4,
 6,
 2,
 12,
 6,
 6,
 10,
 6,
 12,
 2,
 4,
 14,
 12,
 4,
 2,
 4,
 6,
 12,
 2,
 4,
 12,
 12,
 2,
 6,
 4,
 20,
 4,
 2,
 18,
 4,
 6,
 2,
 10,
 2,
 6,
 6,
 10,
 8,
 16,
 2,
 12,
 10,
 2,
 4,
 6,
 6,
 12,
 6,
 6,
 6,
 20,
 4,
 14,
 4,
 2,
 4,
 14,
 6,
 16,
 8,
 6,
 10,
 2,
 10,
 2,
 6,
 12,
 10,
 12,
 6,
 2,
 10,
 8,
 4,
 2,
 18,
 10,
 2,
 6,
 4,
 18,
 6,
 2,
 12,
 10,
 12,
 6,
 2,
 16,
 6,


In [29]:
numadm

array([   2,    2,    1,    1,    3,    2,    2,    2,    5,    4,   10,
          6,    5,    6,   11,   10,   12,    8,   13,    9,   17,   15,
         17,   27,   21,   17,   21,   28,   22,   31,   33,   38,   35,
         38,   41,   49,   54,   49,   55,   60,   64,   71,   66,   74,
         83,   89,   83,   89,   94,   98,  106,  100,  101,  107,  116,
        112,  114,  128,  127,  135,  130,  147,  140,  149,  148,  159,
        151,  167,  179,  180,  187,  196,  196,  203,  201,  218,  220,
        216,  223,  227,  229,  240,  248,  250,  257,  255,  263,  265,
        269,  282,  277,  284,  296,  298,  307,  318,  334,  331,  332,
        340,  344,  351,  351,  363,  358,  378,  383,  395,  399,  401,
        406,  409,  410,  416,  437,  436,  436,  439,  463,  465,  471,
        463,  488,  488,  498,  502,  518,  514,  519,  528,  548,  541,
        546,  554,  564,  571,  581,  586,  587,  597,  598,  608,  606,
        616,  627,  631,  635,  635,  658,  656,  6

In [30]:
Eng459_str_data[0][7]

'[np.float64(73.54103032444594), np.float64(17.396315039874874)]'

In [31]:
# 4. parse the values for log10(winfty) where winfty is broken into its two factors
#  winfty(q <= J+1) and wfinty(q | Q and q> J+1)
str_winf = Eng459_str_data[0][7]
i0=0
while (str_winf[i0] != '('):
    i0 += 1
i1 = i0
while (str_winf[i1] != ')'):
    i1 += 1
logwsinf_J = float(str_winf[(i0+1):i1])
i0 = i1
while (str_winf[i0] != '('):
    i0 += 1
i1 = i0
while (str_winf[i1] != ')'):
    i1 += 1
logwsinf_Q = float(str_winf[(i0+1):i1])


In [32]:
logwsinf_J, logwsinf_Q

(73.54103032444594, 17.396315039874874)

In [33]:
smallp[0:48], smallp[(probelength-5):(probelength+10)]

(array([  2,   3,   5,   7,  11,  13,  17,  19,  23,  29,  31,  37,  41,
         43,  47,  53,  59,  61,  67,  71,  73,  79,  83,  89,  97, 101,
        103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167,
        173, 179, 181, 191, 193, 197, 199, 211, 223]),
 array([81749, 81761, 81769, 81773, 81799, 81817, 81839, 81847, 81853,
        81869, 81883, 81899, 81901, 81919, 81929]))

In [34]:
# read in the primorial coordinate extensions for this constellation, from file
filenamezeros = 'Eng459B_'+str(Engdex)+'_mzeros.csv'
Eng459_m_data = []
with open(filenamezeros, 'r') as fqtr:
    data_reader = csv.reader(fqtr)
    for row in data_reader:
        Eng459_m_data.append([x for x in row])
fqtr.close()

In [35]:
# data: max number of zeros at end of primorial extension, index of this extension in breadth-first search
# len(m_data) yields the number of extensions recorded
len(Eng459_m_data), np.fromstring(Eng459_m_data[0][0][1:-1], dtype=int, sep=',')

(179982, array([       3, 37815854]))

In [ ]:
# looping on 'sdex' through the recorded extensions in 'Eng459_m_data'
# Looking for long runs of mk=0
sdex = 0
Engs = Eng459_sortedB[Engdex]
primorial_probe = np.zeros(probelength, dtype=int)
maxmaxrun0 = 0
maxmaxdex = 0
maxcurrun0 = 0

while (sdex < len(Eng459_m_data)):
    temstr = Eng459_m_data[sdex][1][1:-1]
    sprefix = np.fromstring(temstr, dtype=int, sep=',')

    # the index in smallp[] for the end of the prefix for s
    # smallp[4]=11
    ip0 = len(sprefix)  

    # reset the probe array: copy over the prefix, then zero out the rest of the array
    j = 0
    while (j < ip0):
        primorial_probe[j] = sprefix[j]
        j += 1
    primorial_probe[ip0:] = 0
    
    # Prepare for the opportunistic probe beyond the prefix
    # We want to know whether mk=0 is admissible for this instance
    ip = ip0
    run0 = 0
    maxrun0 = 0

    while ( ip < probelength):
        pk= smallp[ip+4]

        # calculate residue mod pk for the current 
        rez0 = primorial_probe[0] % pk
        primorialres = 2310 % pk
        # print(f"{ip} p {pk} rez0 {rez0}", end='\r')
        j=1
        while (j < ip) :
            rez0 = (rez0 + primorial_probe[j]*primorialres) % pk
            primorialres = (primorialres * smallp[j+4]) % pk    
            j += 1
        # print(f"{ip} p {pk} j{j} r0 {rez0} p#modpk {primorialres}")

        # if mk=0 is an admissible residue, use it.
        # otherwise, pick the smallest mk that yields an admissible residue
        rezlist = list(admissible(pk,Engs))
        if rez0 in rezlist:
            primorial_probe[ip] = int(0)
            run0 += 1  
        else:
            mk = 0
            rezk = rez0
            while (rezk not in rezlist):
                mk += 1
                rezk = (rez0 + mk*primorialres) % pk
            primorial_probe[ip] = int(mk)
            if (run0 > maxrun0):
                maxrun0 = run0
                maxdex = sdex
                print(f"{ip:4d} p {pk:6d} rez0 {rez0:5d} mk {mk:5d} maxrun0 {maxrun0:3}", end='\r')
            run0 = 0
        
        ip += 1

    if (run0 > maxrun0):
        maxrun0 = run0
        print(f"{ip:4d} p {pk:6d} rez0 {rez0:5d} mk {mk:5d} maxrun0 {maxrun0:3}", end='\r')

    if (run0 > maxcurrun0):
        maxcurrun0 = run0

    if (maxrun0 > maxmaxrun0):
        maxmaxrun0 = maxrun0
        maxmaxdex = sdex

    print(f"{sdex:6d} cur {run0:4d} max {maxrun0:4d} {maxmaxrun0:4d} maxcur {maxcurrun0:4d} ")
    sdex += 1


6996 p  70663 rez0 67745 mk     1 maxrun0 490

(array([  2,   3,   5,   7,  11,  13,  17,  19,  23,  29,  31,  37,  41,
         43,  47,  53,  59,  61,  67,  71,  73,  79,  83,  89,  97, 101,
        103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167,
        173, 179, 181, 191, 193, 197, 199, 211, 223]),
 array([75931, 75937, 75941, 75967, 75979, 75983, 75989, 75991, 75997,
        76001, 76003, 76031, 76039, 76079, 76081, 76091, 76099, 76103,
        76123, 76129, 76147, 76157, 76159, 76163, 76207, 76213, 76231,
        76243, 76249, 76253, 76259, 76261, 76283, 76289, 76303, 76333,
        76343, 76367, 76369, 76379, 76387, 76403, 76421, 76423, 76441,
        76463, 76471, 76481, 76487, 76493]))

  51 p    263 rez0   233 mk    13 maxrun0   1
  94 p    523 rez0   281 mk     4 maxrun0   2
 119 p    683 rez0   171 mk     1 maxrun0   3
 132 p    773 rez0   523 mk     1 maxrun0   6
 262 p   1709 rez0  1689 mk     1 maxrun0   7
 294 p   1979 rez0  1300 mk     1 maxrun0   9
 309 p   2083 rez0   493 mk     1 maxrun0  13
 415 p   2903 rez0  2367 mk     1 maxrun0  20
 518 p   3761 rez0  3603 mk     1 maxrun0  29
 830 p   6421 rez0  3679 mk     1 maxrun0  34
1147 p   9311 rez0  6419 mk     1 maxrun0  97
1339 p  11087 rez0  9681 mk     1 maxrun0 101
1731 p  14821 rez0 11669 mk     1 maxrun0 107
2220 p  19609 rez0 19337 mk     1 maxrun0 188
4991 p  48563 rez0 45873 mk     1 maxrun0 442
6639 p  66643 rez0 64775 mk     1 maxrun0 472
7999 p 81853 rez0 1271

In [69]:
run0, maxrun0

(84, 472)

In [70]:
len(primorial_probe), primorial_probe[-100:], smallp[probelength+4], (primorial_probe == 0).sum()

(8000,
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 np.int64(81869),
 np.int64(7656))

In [30]:
temprim = 2310
ip = 5
pk = 76249
while (ip <= 7502):
    temprim = (temprim * smallp[ip]) % pk
    ip += 1

In [31]:
temprim, ip, smallp[7495:7505]

(np.int64(45891),
 7503,
 array([76147, 76157, 76159, 76163, 76207, 76213, 76231, 76243, 76249,
        76253]))

In [32]:
smallp[0:10]

array([ 2,  3,  5,  7, 11, 13, 17, 19, 23, 29])

## Surviving the sieve
For a specific instance $\gamma_0$ of a constellation $s_i(p_0)$ to survive Eratosthenes sieve, $m_k=0$ has to produce
an admissible residue for $s_i$.

In [ ]:
# This function parses the "Eng459_xx_mzeros.csv" files
# and returns a list of prefixes for the primorial coordinates


In [14]:


# ==================================================
# MAIN LOOP through COUNTEREXAMPLES ================
# =========================================================
#  
# =========================================================
debug_verbose = False
num_cons = Eng459_constellations.shape[0]

Eng459_prefixes = []

# For each constellation --
icons = 0

while (icons < num_cons):
    constellation_k = Eng459_constellations[icons]
    gammam_list = []
    
    # Calculate gamma0 in G(11#)
    gamma0 = findgamma11(constellation_k)

    # Calculate residues across ranges of primes p
    # For each prime >= 11, so index i >= 4, we record the list of admissible residues for gamma_0 mod p[i]
    rezlist = []
    i=4
    while (smallp[i] < 250):
        p = smallp[i]
        rezp = admissible(p,constellation_k)
        rezlist.append(rezp)
        i += 1

    # Summarize num_admissible across the primes p
    num_admissible = [len(rezlist[i]) for i in range(len(rezlist))]
    num_admissible = np.array(num_admissible)

    if debug_verbose:
        print(f"{icons:2d} gamma0 {gamma0:4d} num_adm {num_admissible[0:60]}")
        for element in rezlist:
            print(f" {list(element)[0]}", end=' ')
        print()

    # Calculate primorial coordinates for unique prefix
    ip = 1
    gamma_m = [int(gamma0)]

    while (num_admissible[ip] == 1):
        pk = smallp[ip+4]  # primes are offset 4 in array, p0=11 so we start at pk=13
        
        # calculate the residue r0 mod pk
        r0 = gamma_m[0] % pk
        i = 1
        rpml = 2310 % pk  # 11# mod pk
        while (i < ip):
            r0 = (r0 + gamma_m[i]*rpml) % pk
            rpml = (rpml * smallp[i+4]) % pk
            i += 1

        target_r = list(rezlist[ip])[0]
        mk = primorialm(ip,target_r,r0)
        gamma_m.append(int(mk))
        ip += 1  # next prime

    
    # Report and save this information
    Eng459_prefixes.append(gamma_m)

    print(f"{icons:2d} prefix {Eng459_prefixes[icons]}")
    icons += 1   # next constellation - 


 0 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 83, 22, 9, 81, 107, 103, 8, 53]
 1 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 49, 64, 50, 10, 29, 28, 27, 38]
 2 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 68, 47, 36, 52, 96, 79, 84, 99, 92]
 3 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 70, 23, 51, 75, 2, 86, 23, 71, 104]
 4 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 18, 73, 49, 27, 104, 22, 89, 55, 26]
 5 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 70, 52, 33, 55, 75, 37, 98, 107, 74]
 6 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0, 32, 45, 47, 27, 28, 55, 55, 61, 68, 76, 18, 32, 62, 31, 46, 9, 63]
 7 prefix [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 13, 39, 42, 17, 23, 21, 20, 20, 82, 70, 44, 77, 66, 117, 67]
 8 prefix [1271, 5, 8, 9, 17, 21, 29, 13, 2, 8, 0,

In [16]:
Eng_pre_sorted, Eng_consts = zip(*sorted(zip(Eng459_prefixes, Eng459_constellations)))

In [17]:
i=0
while (i < len(Eng_pre_sorted)):
    print(f"{i:2d} {len(Eng_pre_sorted[i])} m {Eng_pre_sorted[i]}")
    print(f" {Eng_consts[i][0:24]}")
    i += 1

 0 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 4, 4, 53, 64, 11, 39, 27, 17, 44, 1, 78, 22, 108, 29, 112, 72]
 [ 2  4 14  4  6  2 10  2  6  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4]
 1 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 22, 73, 67, 66, 95, 17, 109, 111, 99]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 2 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 35, 51, 12, 18, 55, 43, 42, 44, 132]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 3 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 37, 1, 4, 60, 40, 21, 71, 114, 10]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  2  6  4  6]
 4 29 m [107, 6, 8, 9, 5, 7, 1, 23, 38, 34, 46, 20, 13, 5, 60, 51, 70, 42, 25, 33, 50, 80, 51, 11, 0, 47, 4, 47, 43]
 [ 2  4 14  4  6  2 10  8  6 10  6  2 10  6  2 12 10  2  4 12  8  4  6  6]
 5 29 m [107, 6, 8, 9, 5, 7, 1

### Sorted by primorial coordinates
The constellations and their primorial prefixes are now sorted by those prefixes.
This data is saved to file so that we can start further explorations from here.

In [18]:
'''  Commenting out this block to protect the existing files
with open('Eng459_sorted.csv','w',newline='\n') as fp:
    writer = csv.writer(fp)
    writer.writerows(Eng_consts)

with open('Eng459_prefixes.csv','w',newline='\n') as fq:
    writerB = csv.writer(fq)
    writerB.writerows(Eng_pre_sorted)
'''

"  Commenting out this block to protect the existing files\nwith open('Eng459_sorted.csv','w',newline='\n') as fp:\n    writer = csv.writer(fp)\n    writer.writerows(Eng_consts)\n\nwith open('Eng459_prefixes.csv','w',newline='\n') as fq:\n    writerB = csv.writer(fq)\n    writerB.writerows(Eng_pre_sorted)\n"

In [19]:
smallp[0:10],smallp[450:463]

(array([ 2,  3,  5,  7, 11, 13, 17, 19, 23, 29]),
 array([3187, 3191, 3203, 3209, 3217, 3221, 3229, 3251, 3253, 3257, 3259,
        3271, 3299]))

## $\Delta \Phi$ to picture $p$-rough numbers

The count of $p$-rough numbers up through $x$ is denoted $\Phi(x,p)$.
The graph of $\Phi(x,p)$ has a line of symmetry
$$ \tilde{\Phi} = \frac{\phi(p^\#)}{p^\#} x = \frac{1}{\mu} x $$
where $\mu = \frac{p^\#}{\phi(p^\#)}$ is the mean size of the gaps in
$\mathcal{G}(p^\#)$.

So we work with $\Delta \Phi(x,p)$, which measures the deviation of $\Phi(x,p)$ around its line of symmetry.
$$ \Delta \Phi(x,p) = \Phi(x,p) - \frac{1}{\mu} x$$

The deviations of the $p$-rough numbers from their line of symmetry
$$\tilde{\Phi} = \frac{1}{\mu}x$$ 
are periodic and bounded.  Thus the behavior of $\Phi(x,p)$ is completely described by the behavior of $\Delta \Phi(x,p)$
over the first cycle $\mathcal{G}(p^\#)$, or using the rotational symmetry around $x=1+\frac{p^\#}{2}$ over the first
half of this cycle.

$\Delta \Phi(x,p)$ provides a good visualization of the $p$-rough numbers.

To use this visualization here, we use the average prime gap over the first $462$ primes, $p_{462}=3271$

| $k$ | $456$ | $457$ | $458$ | $459$ | $460$ | $461$ | $462$ | $463$ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $p_k$ | $3221$ | $3229$ | $3251$ | $3253$ | $3257$ | $3259$ | $3271$ | $3299$ |
| $\max|s|$ |  | $3236$ | $3240$ | $3242$ | $3276$ | | | |



In [20]:
# average gap size for 
mu_gap = 3271/462
mu_recip = -1/mu_gap
print("mu",mu_gap, "neg reciprocal (slope)", mu_recip)

mu 7.08008658008658 neg reciprocal (slope) -0.14124121063894834


In [21]:
# create the constellation of prime gaps
prime_constellation = smallp[1:462]-smallp[0:461]
prime_constellation = np.concatenate(([2], prime_constellation))


In [22]:
# this function interleaves arr1 with arr2, which we need for plotting the vertical segments
def interleave_np(arr1, arr2):
    stacked_arr = np.stack((arr1, arr2),axis=1)
    return stacked_arr.flatten().tolist()

In [27]:
# Plotting pi function vs Engelsma constellation (459,3242) 
# Interleaving data to get the stepped graph, rendering the vertical segments
# 
def Eng459plot(input_dex):
    input_s = Eng_consts[input_dex][1:]

    # data for plotting the prime constellation
    xp = smallp[0:462] # 462 values from 2 to 3271
    xp = interleave_np(xp,xp)  # ... doubled up...
    xp = np.concatenate(([0],xp)) # 1+2*462 values from 0 to 3271

    delpi = np.zeros(462) # we will calculate the lower points first
    i=1
    delpi[0] = 2*mu_recip
    while (i < 462):
        delpi[i] = delpi[i-1] + 1 + prime_constellation[i]*mu_recip
        i += 1

    delpi = interleave_np(delpi, (delpi + 1)) # interleave the lower points and upper points for the vertical segments
    delpi = np.concatenate(([0],delpi))

    pidf = pd.DataFrame({'x':xp, 'picnt': delpi})

    # data for plotting the input constellation
    xcons = np.cumsum(input_s)
    xcons = interleave_np(xcons, xcons)
    xcons = np.concatenate(([0],xcons))

    delconst = np.zeros(len(input_s))
    delconst[0] = input_s[0]* mu_recip
    i=1
    while (i < len(input_s)):
        delconst[i] = delconst[i-1] + 1 + input_s[i]*mu_recip
        i += 1

    delconst = interleave_np(delconst, (delconst+1))
    delconst = np.concatenate(([0],delconst))

    data_sample = {'x': xcons, 'DelPhi': delconst}
    df = pd.DataFrame(data_sample)

    # plotting the two curves
    plt.clf()
    fig, ax = plt.subplots()
    fig.set_size_inches(14,9)
    ax.set_title(f"Engelsma(459, 3242) #{input_dex} vs $\pi(n)$ as segments of $\Delta \Phi(x,\mu)$")
    ax.grid(axis='y', color='#080408', lw=0.125 )
     
    ax.plot(xcons, delconst, color='#2222AF', lw=0.125, label='DelPhi')
    ax.plot(xp, delpi, color='#AF0000', lw=0.25, label='pi(n)')
# ax.set_ylim(-8,12)

    plt.show()

xEng459Select = widgets.IntSlider(value=29, min=0, max=57, description="Which (459,3242)", 
                                  layout=widgets.Layout(width='80%'), style={'description_width':'90pt'}, disabled=False)

interact(Eng459plot, input_dex=xEng459Select)

interactive(children=(IntSlider(value=29, description='Which (459,3242)', layout=Layout(width='80%'), max=57, …

<function __main__.Eng459plot(input_dex)>

In [53]:
len(Eng459_prefixesB[30])

28

In [57]:
np.log10(len(Eng459_sortedB[30]))

np.float64(2.661812685537261)

## Data files _results and _mzeros
The search through primorial coordinates above produces two sets of output files:  <i>Eng459_xx_results.csv</i> and <i>Eng459_xx_mzeros.csv</i>.  At this point, the search is breadth-first and exhaustive.  So not very deep.  

In order to pursue the survivial of any incidence of these (459,3242)-counterexamples, we have to switch to a greedy depth-first
approach.  Survival is indicated by a <i>long</i> sequence of consecutive primorial coordinates $m_k = 0$.  So we start by considering
those constellations with primorial expansions from the current search that end with at least one $m_k=0$.

<i>Eng459_xx_results.csv</i> contains summary notes about the search over the (459,3242)-counterexample of index <i>xx</i> in the file 
<i>Eng459_sorted.csv</i>  The first few fields in the <i>_results</i> file are the index <i>xx</i> of the counterexample; the length of
the unique prefix of the primorial coordinates, starting at $p_{0}=11$; the prime $p_{k_0}$ just beyond this unique prefix, where there is more than one admissible residue; the depth $k$ of the breadth-first search into extensions of the prefix; the prime corresponding to 
the last $m_k$ recorded; and the number of admissible instances searched.  Then the array of the numbers of admissible instances $\bmod p$
is listed, starting at $p_{k_0}$.  After this array we list the two factors (in log-base-10) of the asymptotic relative population
for this constellation:
$$ w_{s,J}(\infty) \; = \; \prod_{p \le J+1} (p - \nu(p)) \cdot \prod_{p > J+1} \frac{p - \nu(p)}{p - J-1}$$

<i>Eng459_xx_mzeros.csv</i> contains data about the extensions of the primorial coordinates for the (459,3242)-counterexample of
index <i>xx</i> in the file <i>Eng459_sorted.csv</i>  Each row of data starts with a two-element array:  
the number of terminal zeroes for this extension, and the index of the extension in the breadth-first search.  
This is followed by the primorial coordinates from $p_0=11$.

In [87]:
# We start with the unique prefixes in Eng_pre_sorted and the constellations in Eng_const
# The array smallp starts at p=2, and p0 for the prefixes is p=11.  The primorial coordinate m[k] corresponds to smallp[k+4]

debug_verbose = True
ic = int(0)  # index for the constellation

while (ic < len(Eng459_sortedB)):
    pref_len = len(Eng459_prefixesB[ic])  # for the ic-th counterexample - 
    Eng459s = Eng459_sortedB[ic]             # get the constellation
    mprefix = Eng459_prefixesB[ic]        # get the unique prefix for the primorial coordinates

    # for this constellation and prefix, determine the admissible residues for an exhaustive search 
    #  of up to 20M examples each
    rezlist = []
    ip0 = 4 + pref_len
    while (smallp[ip0] < 5000): 
        p = smallp[ip0]
        rezp = admissible(p,Eng459s)
        rezlist.append(rezp)
        ip0 += 1

    # Summarize num_admissible across the primes p
    num_admissible = [len(rezlist[i]) for i in range(len(rezlist))]
    num_admissible = np.array(num_admissible)

    if debug_verbose:
        print(f"{ic:2d} gamma0 {Eng459s[0]:4d} len_prefix {pref_len:2d} num_adm {num_admissible[0:60]}")
        # for element in rezlist:
            # print(f" {list(element)[0]}", end=' ')
        # print()

    # We extend the primorial coordinates 
    num_instances = 1
    m_ext_list = []
    k = 0      # index for pk beyond the prefix, e.g. for num_admissible and rezlist

    
    if debug_verbose:
        print(f"k {k} num_admissible {num_admissible[k]} total instances {num_instances}", end='\r')

    # initiate the extensions in first iteration on k
    ik = k+pref_len   # index for smallp[] corresponding to k
    pk = smallp[ik+4]
    num_instances = num_admissible[k]
    i = 0
    while (i < num_admissible[k]):
        m_ext_list.append(list(mprefix))

        target_residue = list(rezlist[k])[i]
        # calculate residue r0 modulo pk
        # set initial conditions for this loop
        j = 1
        r0 = m_ext_list[i][0] % pk
        rpml = 2310 % pk  # initialize this factor at 11#
        while (j < ik):  # calculate the residue for the primorial expansion
            r0 = (r0 + m_ext_list[i][j] * rpml) % pk
            rpml = (rpml * smallp[j+4]) % pk
            j += 1
        # calculate next primorial coefficient mk
        mk = primorialm(ik,target_residue,r0)
        (m_ext_list[i]).append(int(mk))

        if debug_verbose:
            print(f"k {k} {ik} p {smallp[ik+4]} i {i} targetr {target_residue} r0 {r0}")
            for element in m_ext_list[i]:
                print(f"{element:3d}", end=" ")
            print()

        i += 1

    # now iterate k through the primes pk.  
    #  k is the index in num_admissible[] which starts after the unique prefix
    #  ik is the corresponding index in smallp[], as an offset from 11
    k += 1
    ik += 1

    while (num_instances < 4000000):   
        num_instances *= num_admissible[k]
        pk = smallp[ik+4]
    
        iprefix=0
        num_prefixes = len(m_ext_list)
        # iprefix is the index through existing primorial expansions
        # i is the index through admissible residues at this stage
        
        while (iprefix < num_prefixes):
            # calculate r0 mod pk
            j = 1
            r0 = m_ext_list[iprefix][0] % pk
            rpml = 2310 % pk  # initialize this factor at 11#
            while (j < ik):  # the expansion for r 
                r0 = (r0 + m_ext_list[iprefix][j] * rpml) % pk
                rpml = (rpml * smallp[j+4]) % pk
                j += 1

            # extend the first copy in place
            # save the prefix
            m_start = list(m_ext_list[iprefix].copy())
            # calculate next primorial coefficient mk
            target_residue = list(rezlist[k])[0]
            mk = primorialm(ik,target_residue,r0)
            (m_ext_list[iprefix]).append(int(mk))

            if ((iprefix % 4096)==0):
                print(f"ic {ic} k {k} {ik} p {pk} i {iprefix} targetr {target_residue} r0 {r0} length {len(m_ext_list[iprefix])}", end='\r')
        
            i=1
            while (i < num_admissible[k]):
                icopy = len(m_ext_list)
                m_ext_list.append(m_start.copy())  # icopy is the index for this copy
                target_residue = list(rezlist[k])[i]
                mk = primorialm(ik,target_residue,r0)
                (m_ext_list[icopy]).append(int(mk))
                # print(f"Copy {icopy} k {k} p {smallp[k+4]} i {iprefix} {i} targetr {target_residue} r0 {r0} length {len(gammam_list[icopy])}")

                i += 1

            iprefix += 1  # next prefix

        k += 1     # next prime - depth of search
        ik += 1

    # Process and record the results for this constellation
    filename = 'Eng459_' + str(ic) + '_results.csv'
    k -= 1
    ik -= 1
    
    # record ic, prefix_len, p0, k, pk, num_instances, num_admissible
    # XXXQHERE [6/22] - how much of num_admissible to save?
    results_list = [ic, pref_len, smallp[4+pref_len], k, smallp[k+pref_len+4], num_instances, num_admissible]
    
    # Factors of asymptotic relative population 
    # - up through J+1=460, and up through |s|/2 = 1621
    # We calculate the log of this factor, anticipating floating point overflow
    ws_J_log = 0
    i = 0    # indexing here: admissible 0 == primes 4+pref_len
    pk = smallp[i + 4 + pref_len]
    while (pk <= 460):
        ws_J_log += np.log10(num_admissible[i])
        i += 1
        pk = smallp[i + 4 + pref_len]

    ws_s_log = 0
    while (pk <= 1621):
        ws_s_log += np.log10(num_admissible[i] / (pk-460))
        i += 1
        pk = smallp[i + 4 + pref_len]

    results_list.append([ws_J_log, ws_s_log])

    with open(filename, 'w', newline='\n') as fptr:
        writer = csv.writer(fptr)
        writer.writerow(results_list)
    fptr.close()
    
    # Identify instances whose primorial coordinates end in sequences of 0's
    num_mext = len(m_ext_list)
    zero_list = []
    imext = 0
    while (imext < num_mext):
        j = len(m_ext_list[imext])-1
        while ( m_ext_list[imext][j] == 0):
            j -= 1
        nzer = len(m_ext_list[imext]) - 1 - j
        if (nzer > 0):
            zero_list.append([nzer,imext])
        imext += 1

    if (len(zero_list)>0):
        print(f"Saving {len(zero_list)} extensions out of {num_mext}")
        sorted_zero_list = sorted(zero_list, reverse=True)
        max_zeros = sorted_zero_list[0][0]
        m_zeros = [] 
        j=0
        while (j < len(sorted_zero_list)):
            imext = sorted_zero_list[j][1]
            entry = [ sorted_zero_list[j], m_ext_list[imext]]
            m_zeros.append(entry)
            j += 1

        filename = 'Eng459_' + str(ic) + '_mzeros.csv'

        with open(filename, 'w', newline='\n') as fptr:
            writer = csv.writer(fptr)
            writer.writerows(m_zeros)
        fptr.close()


    ic +=1   # Loop into next constellation (459,3242)



 0 gamma0    2 len_prefix 29 num_adm [  2   1   1   3   1   2   3   4   3   9   5   5   5  10   8  12   7   9
   9  18  14  17  25  21  17  22  27  21  28  31  37  34  39  38  50  52
  48  54  59  64  72  68  73  80  89  85  87  94  95 103 103 102 108 112
 112 115 132 126 134 129]
k 0 29 p 139 i 0 targetr 24 r0 91ces 1
107   6   8   9   5   7   1  23  38  34  46  20  13   4   4  53  64  11  39  27  17  44   1  78  22 108  29 112  72 132 
k 0 29 p 139 i 1 targetr 90 r0 91
107   6   8   9   5   7   1  23  38  34  46  20  13   4   4  53  64  11  39  27  17  44   1  78  22 108  29 112  72 114 
Saving 22910 extensions out of 48600002 r0 6 length 4343
 1 gamma0    2 len_prefix 29 num_adm [  2   1   1   3   2   3   2   4   3   9   5   5   6  11   8  12   8   9
  10  17  16  16  24  19  18  21  27  21  29  32  38  35  39  39  48  49
  49  56  60  66  69  65  73  82  88  81  89  94  96 104  99 102 109 113
 110 115 129 123 133 130]
k 0 29 p 139 i 0 targetr 24 r0 136es 1
107   6   8   9   5   7  

In [85]:
m_ext_list[1000],m_ext_list[1001], num_mext

([107,
  6,
  8,
  9,
  5,
  7,
  1,
  23,
  38,
  34,
  46,
  20,
  13,
  4,
  4,
  53,
  64,
  11,
  39,
  27,
  17,
  44,
  1,
  78,
  22,
  108,
  29,
  112,
  72,
  np.int64(5),
  np.int64(5),
  np.int64(6),
  np.int64(12),
  np.int64(18),
  np.int64(27),
  np.int64(21),
  np.int64(3),
  np.int64(32),
  np.int64(36),
  np.int64(24),
  np.int64(42),
  np.int64(14),
  np.int64(1)],
 [107,
  6,
  8,
  9,
  5,
  7,
  1,
  23,
  38,
  34,
  46,
  20,
  13,
  4,
  4,
  53,
  64,
  11,
  39,
  27,
  17,
  44,
  1,
  78,
  22,
  108,
  29,
  112,
  72,
  np.int64(5),
  np.int64(5),
  np.int64(6),
  np.int64(12),
  np.int64(18),
  np.int64(27),
  np.int64(21),
  np.int64(3),
  np.int64(32),
  np.int64(12),
  np.int64(24),
  np.int64(42),
  np.int64(14),
  np.int64(1)],
 4860000)

In [38]:
# This is a list of the NUMBER of admissible residues mod p starting at p=11 for the counterexample
num_admissible

array([   1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    2,    2,    1,    1,    3,
          1,    2,    2,    5,    4,   10,    6,    5,    6,   13,   10,
         12,    8,   13,    9,   17,   15,   17,   26,   21,   17,   21,
         29,   23,   31,   32,   38,   37,   38,   41,   47,   53,   49,
         55,   59,   62,   68,   66,   74,   82,   88,   83,   89,   95,
         97,  105,   99,  100,  109,  114,  112,  116,  128,  128,  135,
        129,  144,  141,  148,  148,  157,  153,  166,  178,  180,  185,
        196,  194,  203,  201,  218,  219,  215,  221,  227,  228,  238,
        246,  248,  255,  255,  262,  265,  268,  281,  276,  284,  296,
        297,  307,  318,  334,  332,  331,  339,  343,  350,  349,  362,
        357,  378,  383,  393,  400,  399,  405,  408,  409,  415,  438,
        435,  436,  438,  463,  464,  470,  463,  4

In [39]:
num_459admissible = np.load('Eng459numadm.npy')
num_459admissible[0:100]

array([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1,   1,   2,   2,   1,   1,   3,   1,   2,   2,   5,   4,  10,
         6,   5,   6,  13,  10,  12,   8,  13,   9,  17,  15,  17,  26,
        21,  17,  21,  29,  24,  31,  32,  39,  37,  38,  41,  48,  53,
        49,  55,  59,  63,  69,  66,  74,  82,  89,  83,  89,  95,  98,
       105, 100, 100, 109, 115, 112, 116, 128, 128, 135, 129, 145, 141,
       149, 148, 158, 153, 166, 179, 181, 186, 197])

In [40]:
i=0
while (num_admissible[i] == num_459admissible[i]):
    i += 1
print(f"{i} p {smallp[i+5]} 458L: {num_admissible[i]} 459: {num_459admissible[i]}")

56 p 293 458L: 23 459: 24


In [51]:
# Develop the primorial coordinates for our Engelsma counterexample, noting the length of the driving term as the sieve progresses
k = 1
gamma_expansion = [1271]         # start the expansion with gamma0 in G(11#)
constellationk = Engelsma11L.copy()
while (num_admissible[k] == 1 and k < len(rezlist)):  # unique admissible instance for p
    pk = smallp[k+4]
    target_residue = list(rezlist[k])[0]
    # calculate residue r0 modulo pk
    # set initial conditions for this loop
    i = 1
    r0 = gamma_expansion[0] % pk
    rpml = 2310 % pk  # initialize this factor at 11#
    while (i < k):  # the expansion for r nests from right to left
        r0 = (r0 + gamma_expansion[i] * rpml) % pk
        rpml = (rpml * smallp[i+4]) % pk
        i += 1
    # calculate next primorial coefficient mk
    mk = primorialm(k,target_residue,r0)
    gamma_expansion.append(mk)
    # perform any interior fusions indicated for this image of s
    r0 = target_residue
    i=0
    while (i < len(constellationk)):
        r0 = (r0+constellationk[i]) % pk
        if (r0 == 0):
            constellationk[i] = constellationk[i] + constellationk[i+1]
            r0 = constellationk[i+1] % pk
            constellationk[i+1] = 0
        i += 1
    constellationk = constellationk[ constellationk != 0]
    print(f"k {k:3d} pk {pk:3d} mk {mk:3d} r {target_residue:3d} length {len(constellationk)}  ") # , end='\r')
    k += 1

k   1 pk  13 mk   5 r   3 length 623  
k   2 pk  17 mk   8 r  16 length 590  
k   3 pk  19 mk   9 r  10 length 566  
k   4 pk  23 mk  17 r  12 length 545  
k   5 pk  29 mk  21 r  25 length 529  
k   6 pk  31 mk  29 r  16 length 517  
k   7 pk  37 mk  13 r   3 length 507  
k   8 pk  41 mk   2 r  37 length 496  
k   9 pk  43 mk   8 r  22 length 490  
k  10 pk  47 mk   0 r  24 length 486  
k  11 pk  53 mk  32 r   9 length 480  
k  12 pk  59 mk  45 r  30 length 479  
k  13 pk  61 mk  47 r  22 length 475  
k  14 pk  67 mk  27 r  34 length 472  
k  15 pk  71 mk  28 r  36 length 470  
k  16 pk  73 mk  27 r  34 length 467  
k  17 pk  79 mk  24 r  40 length 467  
k  18 pk  83 mk   0 r  42 length 467  
k  19 pk  89 mk   9 r  38 length 465  
k  20 pk  97 mk  15 r  69 length 463  
k  21 pk 101 mk  68 r  76 length 462  
k  22 pk 103 mk 100 r  52 length 462  
k  23 pk 107 mk  53 r  54 length 462  
k  24 pk 109 mk  45 r  76 length 461  
k  25 pk 113 mk   7 r 100 length 460  
k  26 pk 127 mk  24 r  64

In [52]:
# The above search was over the sequence of primes for which there is a unique admissible residue
# This is the prefix for any further primorial coordinates for (459,3242)
gammam_prefix = np.array(gamma_expansion)
gammam_prefix

array([1271,    5,    8,    9,   17,   21,   29,   13,    2,    8,    0,
         32,   45,   47,   27,   28,   27,   24,    0,    9,   15,   68,
        100,   53,   45,    7,   24,    5])

In [55]:
rezlist[0]

{6}

## Counterexample surviving the sieve
The Engelsma counterexample (459,3242) has a unique image up through ${\mathcal G}(131^\#)$.  The constellation itself first appears in 
${\mathcal G}(113^\#)$, and there are no longer driving terms.

By ${\mathcal G}(457^\#)$ there are $2.10278720 \; E73$ images of this constellation.
Over $461 \le p_k \le 1619$ the relative population is 
$$\prod_{461}^{1619} \frac{q-\nu(q)}{q-460} \; = \; 2.51736042 \;E17$$

Thus the asymptotic relative population of the Engelsma counterexample (459,3242) is
$$w_{s,459}(\infty) = 5.29347327 \cdot E90$$

In [56]:
# calculating the relative population of the counterexample up through pk=457 
k=27
ns = 1
while (smallp[k+4] < 458):
    ns *= int(num_admissible[k])
    # print(f"k {k} p {smallp[k+4]} adm {num_admissible[k]} n_s {ns:.8e}") 
    k += 1
print(f"k {k-1} p {smallp[k+3]} n_s {ns:.8e}")

k 83 p 457 n_s 1.79093169e+73


In [57]:
print(f"s length {len(constellationk)} span {np.sum(constellationk)}")

s length 460 span 3276


In [58]:
# calculating the relative population of the counterexample over the range from p=457 up through pk=1621
# this is the 
k=84
ws = 1
while (smallp[k+4] <= 1621):
    ws *= float(num_admissible[k])/(smallp[k+4]-460)
    print(f"k {k:3d} p {smallp[k+4]:4d} p-J-1 {(smallp[k+4]-460):4d} adm {num_admissible[k]:4d} w_s {ws:.8e}") 
    k += 1
print(f"k {k-1} p {smallp[k+3]} w_s {ws:.12e}")

k  84 p  461 p-J-1    1 adm  116 w_s 1.16000000e+02
k  85 p  463 p-J-1    3 adm  128 w_s 4.94933333e+03
k  86 p  467 p-J-1    7 adm  128 w_s 9.05020952e+04
k  87 p  479 p-J-1   19 adm  135 w_s 6.43041203e+05
k  88 p  487 p-J-1   27 adm  129 w_s 3.07230797e+06
k  89 p  491 p-J-1   31 adm  144 w_s 1.42713661e+07
k  90 p  499 p-J-1   39 adm  141 w_s 5.15964773e+07
k  91 p  503 p-J-1   43 adm  148 w_s 1.77587875e+08
k  92 p  509 p-J-1   49 adm  148 w_s 5.36387868e+08
k  93 p  521 p-J-1   61 adm  157 w_s 1.38053927e+09
k  94 p  523 p-J-1   63 adm  153 w_s 3.35273822e+09
k  95 p  541 p-J-1   81 adm  166 w_s 6.87104376e+09
k  96 p  547 p-J-1   87 adm  178 w_s 1.40579976e+10
k  97 p  557 p-J-1   97 adm  180 w_s 2.60870058e+10
k  98 p  563 p-J-1  103 adm  185 w_s 4.68553017e+10
k  99 p  569 p-J-1  109 adm  196 w_s 8.42535700e+10
k 100 p  571 p-J-1  111 adm  194 w_s 1.47253987e+11
k 101 p  577 p-J-1  117 adm  203 w_s 2.55491961e+11
k 102 p  587 p-J-1  127 adm  201 w_s 4.04361292e+11
k 103 p  593

In [60]:
# XXXQHERE [6/27] - move the following material into '23_'
# developing some intuition about why the numbers of admissible residues are so high
res = np.zeros(460)
pk = smallp[80]
i=1
while (i < 460):
    res[i] = (res[i-1]+EngelsmaL[i-1]) % pk
    i += 1
covered_res, res_counts = np.unique(res, return_counts=True)
if (pk > 460):
    wsfactor = (pk-len(covered_res))/(pk-460)
else:
    wsfactor = pk - len(covered_res)
print(f"pk {pk} num covered = {len(covered_res)} num admissible = {pk-len(covered_res)} w_s factor {wsfactor:.8f}")
print(covered_res)
print(res_counts)

pk 419 num covered = 324 num admissible = 95 w_s factor 95.00000000
[  0.   1.   2.   3.   4.   5.   6.   7.   8.   9.  12.  13.  14.  15.
  16.  17.  19.  20.  21.  22.  24.  25.  26.  28.  30.  31.  32.  33.
  34.  35.  36.  37.  38.  39.  40.  42.  43.  44.  45.  46.  48.  49.
  50.  51.  52.  53.  54.  56.  57.  62.  63.  66.  68.  69.  71.  72.
  73.  74.  75.  78.  79.  82.  83.  84.  85.  87.  88.  89.  90.  91.
  92.  93.  94.  95.  96.  97.  98.  99. 100. 102. 103. 104. 107. 108.
 109. 110. 112. 113. 114. 115. 116. 117. 118. 119. 120. 122. 123. 124.
 126. 127. 128. 129. 130. 131. 132. 133. 134. 135. 136. 138. 139. 142.
 143. 144. 145. 146. 147. 148. 149. 150. 152. 153. 154. 156. 157. 158.
 159. 160. 161. 162. 163. 164. 165. 168. 169. 170. 171. 172. 173. 174.
 175. 176. 177. 179. 181. 182. 183. 184. 185. 186. 187. 188. 189. 190.
 191. 192. 193. 194. 195. 198. 199. 200. 201. 202. 204. 205. 206. 210.
 212. 213. 214. 215. 216. 217. 218. 219. 220. 221. 222. 223. 224. 225.
 226. 228

In [61]:
smallp[80:90]

array([419, 421, 431, 433, 439, 443, 449, 457, 461, 463])

In [62]:
# calculate the constellation among small primes
del_smallp = smallp[1:501]-smallp[0:500]
del_smallp = np.array([2]+list(del_smallp))
del_smallp[0:100]

array([ 2,  1,  2,  2,  4,  2,  4,  2,  4,  6,  2,  6,  4,  2,  4,  6,  6,
        2,  6,  4,  2,  6,  4,  6,  8,  4,  2,  4,  2,  4, 14,  4,  6,  2,
       10,  2,  6,  6,  4,  6,  6,  2, 10,  2,  4,  2, 12, 12,  4,  2,  4,
        6,  2, 10,  6,  6,  6,  2,  6,  4,  2, 10, 14,  4,  2,  4, 14,  6,
       10,  2,  4,  6,  8,  6,  6,  4,  6,  8,  4,  8, 10,  2, 10,  2,  6,
        4,  6,  8,  4,  2,  4, 12,  8,  4,  8,  4,  6, 12,  2, 18])

In [36]:
# Comparing the counts of gaps in the counterexample vs the counts among the small primes
Eng_gaps, Eng_cnts = np.unique(Engelsma, return_counts=True)
for element in Eng_gaps:
    print(f"{element:3d}",end=" ")
print()
for element in Eng_cnts:
    print(f"{element:3d}",end=" ")
print()

  2   4   6   8  10  12  14  16  18  20  22  24  30 
 88  76 120  41  56  34  15  10  12   2   3   1   1 


In [37]:
pi_gaps, pi_cnts = np.unique(del_smallp[0:459], return_counts=True)
for element in pi_gaps:
    print(f"{element:3d}",end=" ")
print()
for element in pi_cnts:
    print(f"{element:3d}",end=" ")
print()

  1   2   4   6   8  10  12  14  16  18  20  22  24  26  28  34 
  1  86  92 112  44  43  32  18   8   9   3   5   2   2   1   1 


In [38]:
del_smallp[0:459]

array([ 2,  1,  2,  2,  4,  2,  4,  2,  4,  6,  2,  6,  4,  2,  4,  6,  6,
        2,  6,  4,  2,  6,  4,  6,  8,  4,  2,  4,  2,  4, 14,  4,  6,  2,
       10,  2,  6,  6,  4,  6,  6,  2, 10,  2,  4,  2, 12, 12,  4,  2,  4,
        6,  2, 10,  6,  6,  6,  2,  6,  4,  2, 10, 14,  4,  2,  4, 14,  6,
       10,  2,  4,  6,  8,  6,  6,  4,  6,  8,  4,  8, 10,  2, 10,  2,  6,
        4,  6,  8,  4,  2,  4, 12,  8,  4,  8,  4,  6, 12,  2, 18,  6, 10,
        6,  6,  2,  6, 10,  6,  6,  2,  6,  6,  4,  2, 12, 10,  2,  4,  6,
        6,  2, 12,  4,  6,  8, 10,  8, 10,  8,  6,  6,  4,  8,  6,  4,  8,
        4, 14, 10, 12,  2, 10,  2,  4,  2, 10, 14,  4,  2,  4, 14,  4,  2,
        4, 20,  4,  8, 10,  8,  4,  6,  6, 14,  4,  6,  6,  8,  6, 12,  4,
        6,  2, 10,  2,  6, 10,  2, 10,  2,  6, 18,  4,  2,  4,  6,  6,  8,
        6,  6, 22,  2, 10,  8, 10,  6,  6,  8, 12,  4,  6,  6,  2,  6, 12,
       10, 18,  2,  4,  6,  2,  6,  4,  2,  4, 12,  2,  6, 34,  6,  6,  8,
       18, 10, 14,  4,  2

In [39]:
# comparing average gap sizes for small primes, for the counterexample, and for the cycle G(113#)
print(f"pi avg {(3251/458):.4f} vs mu_Eng {(3242/459):.4f} vs mu_113 {(1/mu_recip):.4f}")

pi avg 7.0983 vs mu_Eng 7.0632 vs mu_113 8.7131


In [40]:
mu_gaps = np.zeros(100)
mu_gaps[0] = 2
i=1
while (i<100):
    mu_gaps[i] = mu_gaps[i-1] * (smallp[i]/(smallp[i]-1))
    i += 1

In [41]:
mu_gaps[0:20]

array([2.        , 3.        , 3.75      , 4.375     , 4.8125    ,
       5.21354167, 5.53938802, 5.8471318 , 6.11291052, 6.33122875,
       6.54226971, 6.72399942, 6.89209941, 7.05619701, 7.2095926 ,
       7.34823861, 7.47493238, 7.59951459, 7.71465875, 7.82486816])

In [42]:
smallp[444:470]

array([3121, 3137, 3163, 3167, 3169, 3181, 3187, 3191, 3203, 3209, 3217,
       3221, 3229, 3251, 3253, 3257, 3259, 3271, 3299, 3301, 3307, 3313,
       3319, 3323, 3329, 3331])

In [43]:
smallp[0:5]

array([ 2,  3,  5,  7, 11])

In [44]:
# searching for instances that could survive the sieve.  Far too many copies to search exhaustively, so we pursue a greedy random
# algorithm.  Search randomly for an mk=0, then search greedily for additional 0's
nprimes = len(smallp)
print(f" nprimes {nprimes} maxp {smallp[nprimes-1]}")

 nprimes 646030 maxp 9699691


In [45]:
# random probes through the admissible extensions from the prefixes in gammam_list
# 
k0 = len(gammam_list[0])  # first open index for pk.  Remember the shift pk=smallp[k+4].  smallp[4]=11
k1 = len(rezlist)
print(f"k {k0}-{k1} pmax {smallp[k1+4]}")
logpml = np.log10(2310)
i=5
while (i < (k1+4)):
    logpml = logpml + np.log10(smallp[i])
    i += 1
print(f"i {i-1} p {smallp[i-1]} log(pml) {logpml}")

k 37-665 pmax 5003
i 668 p 4999 log(pml) 2133.1221880361404


In [46]:
# XXXQHERE [21 May] -- random extensions
# opportunistic search:  random search until we find an mk=0, then search for adjacent mk=0
ntries = 2 # 000000
itry = 0
while (itry < ntries):
    iprefix = random.randrange(len(gammam_list))
    test_gamma = gammam_list[iprefix].copy()
    # randomly extend this prefix until we find an m=0

    # try to extend with another 0

    itry += 1

In [ ]:
# Repeating the analysis for case (458,3240)


In [50]:
# For the primorial coordinates for the counterexample we start in G(11#), so we start with the unique driving term in that cycle
Engelsma458_11 = np.array([2,4,2,4,6,2,6,4,2,4,6,6,2,6,6,6,4,6,8,4,2,4,2,4,8,6,4,8,4,6,2,6,6,4,2,4,6,8,4,2,4,2,10,2,10,2,4,2,4,6,2,10,2,4,6,8,6,4,2,6,4,6,8,4,6,2,4,
              8,6,4,6,2,4,6,2,6,6,4,6,6,2,6,6,4,2,10,2,10,2,4,2,4,6,2,6,4,2,10,6,2,6,4,2,6,4,6,8,4,2,4,2,12,6,4,6,2,4,6,2,12,4,2,4,8,6,4,2,4,2,10,2,10,6,2,
              4,6,2,6,4,2,4,6,6,2,6,4,2,10,6,8,6,4,2,4,8,6,4,6,2,4,6,2,6,6,6,4,6,2,6,4,2,4,2,10,12,2,4,2,10,2,6,4,2,4,6,6,2,10,2,6,4,14,4,2,4,2,4,8,6,4,
              6,2,4,6,2,6,6,4,2,4,6,2,6,4,2,4,12,2,12,4,2,4,6,2,6,4,2,4,6,6,2,6,4,2,6,4,6,8,4,2,4,2,4,14,4,6,2,10,2,6,6,4,2,4,6,2,10,2,4,2,12,10,2,4,2,
              4,6,2,6,4,6,6,6,2,6,4,2,6,4,6,8,4,2,4,6,8,6,10,2,4,6,2,6,6,4,2,4,6,2,6,4,2,6,10,2,10,2,4,2,4,6,8,4,2,4,12,2,6,4,2,6,4,6,12,2,4,2,4,8,6,4,6,2,
              4,6,2,6,10,2,4,6,2,6,4,2,4,2,10,2,10,2,4,6,6,2,6,6,4,6,6,2,6,4,2,6,4,6,8,4,2,6,4,8,6,4,6,2,4,6,8,6,4,2,10,2,6,4,2,4,2,10,2,10,2,4,2,4,8,6,4,
              2,4,6,6,2,6,4,8,4,6,8,4,2,4,2,4,8,6,4,6,6,6,2,6,6,4,2,4,6,2,6,4,2,4,2,10,2,10,2,6,4,6,2,6,4,2,4,6,6,8,4,2,6,10,8,4,2,4,2,4,8,10,6,2,4,8,6,
              6,4,2,4,6,2,6,4,6,2,10,2,10,2,4,2,4,6,2,6,4,2,4,6,6,2,6,6,6,4,6,8,4,2,4,2,4,8,6,4,8,4,6,2,6,6,4,2,4,6,8,4,2,4,2,10,2,10,2,4,2,4,6,2,10,2,
              4,6,8,6,4,2,6,4,6,8,4,6,2,4,8,6,4,6,2,4,6,2,6,6,4,6,6,2,6,6,4,2,10,2,10,2,4,2,4,6,2,6,4,2,10,6,2,6,4,2,6,4,6,8,4,2,4,2,12,6,4,6,2,4,6,2,
              12,4,2,4,8,6,4,2,4,2,10,2,10,6,2,4,6,2,6,4,2,4,6,6,2,6,4,2,10,6,8,6,4,2,4,8,6,4,6,2,4,6,2,6,6,6,4,6,2,6,4,2,4,2,10,12,2,4,2,10,2,6,4,2,4,6,
              6,2,10,2,6,4,14,4], dtype=int)
print(f"Driving term for Engelsma ({len(Engelsma458_11)},{np.sum(Engelsma458_11)}) counterexample")

Driving term for Engelsma (673,3240) counterexample


In [29]:
smallp[41]

np.int64(181)

In [30]:
smallp[37:42]

array([163, 167, 173, 179, 181])

In [38]:
smallp[250:260]

array([1597, 1601, 1607, 1609, 1613, 1619, 1621, 1627, 1637, 1657])

In [49]:
del_smallp[0:459]

array([ 2,  1,  2,  2,  4,  2,  4,  2,  4,  6,  2,  6,  4,  2,  4,  6,  6,
        2,  6,  4,  2,  6,  4,  6,  8,  4,  2,  4,  2,  4, 14,  4,  6,  2,
       10,  2,  6,  6,  4,  6,  6,  2, 10,  2,  4,  2, 12, 12,  4,  2,  4,
        6,  2, 10,  6,  6,  6,  2,  6,  4,  2, 10, 14,  4,  2,  4, 14,  6,
       10,  2,  4,  6,  8,  6,  6,  4,  6,  8,  4,  8, 10,  2, 10,  2,  6,
        4,  6,  8,  4,  2,  4, 12,  8,  4,  8,  4,  6, 12,  2, 18,  6, 10,
        6,  6,  2,  6, 10,  6,  6,  2,  6,  6,  4,  2, 12, 10,  2,  4,  6,
        6,  2, 12,  4,  6,  8, 10,  8, 10,  8,  6,  6,  4,  8,  6,  4,  8,
        4, 14, 10, 12,  2, 10,  2,  4,  2, 10, 14,  4,  2,  4, 14,  4,  2,
        4, 20,  4,  8, 10,  8,  4,  6,  6, 14,  4,  6,  6,  8,  6, 12,  4,
        6,  2, 10,  2,  6, 10,  2, 10,  2,  6, 18,  4,  2,  4,  6,  6,  8,
        6,  6, 22,  2, 10,  8, 10,  6,  6,  8, 12,  4,  6,  6,  2,  6, 12,
       10, 18,  2,  4,  6,  2,  6,  4,  2,  4, 12,  2,  6, 34,  6,  6,  8,
       18, 10, 14,  4,  2

In [51]:
smallp[50:89]

array([233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307,
       311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383,
       389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461])

In [54]:
i = 1
w=1.0
primpk = np.ones(200, dtype=float)
primpk[0] = 2.0
while (i < 130):
    # w *= (smallp[i]-1)
    primpk[i] = smallp[i] * primpk[i-1]
    print(f"{i:3d} p {smallp[i]:4d} p# {primpk[i]}")
    i +=1
# switch to log-primorial
primpk[i]= np.log10(primpk[i-1]) + np.log10(smallp[i])
print(f"{i:3d} p {smallp[i]:4d} log-p# {primpk[i]}")
i += 1
while (i < 200):
    primpk[i]= primpk[i-1] + np.log10(smallp[i])
    print(f"{i:3d} p {smallp[i]:4d} log-p# {primpk[i]}")
    i += 1
    

  1 p    3 p# 6.0
  2 p    5 p# 30.0
  3 p    7 p# 210.0
  4 p   11 p# 2310.0
  5 p   13 p# 30030.0
  6 p   17 p# 510510.0
  7 p   19 p# 9699690.0
  8 p   23 p# 223092870.0
  9 p   29 p# 6469693230.0
 10 p   31 p# 200560490130.0
 11 p   37 p# 7420738134810.0
 12 p   41 p# 304250263527210.0
 13 p   43 p# 1.308276133167003e+16
 14 p   47 p# 6.148897825884914e+17
 15 p   53 p# 3.2589158477190046e+19
 16 p   59 p# 1.9227603501542128e+21
 17 p   61 p# 1.1728838135940697e+23
 18 p   67 p# 7.858321551080267e+24
 19 p   71 p# 5.57940830126699e+26
 20 p   73 p# 4.072968059924903e+28
 21 p   79 p# 3.2176447673406735e+30
 22 p   83 p# 2.670645156892759e+32
 23 p   89 p# 2.3768741896345556e+34
 24 p   97 p# 2.3055679639455188e+36
 25 p  101 p# 2.328623643584974e+38
 26 p  103 p# 2.398482352892523e+40
 27 p  107 p# 2.5663761175949998e+42
 28 p  109 p# 2.79734996817855e+44
 29 p  113 p# 3.1610054640417614e+46
 30 p  127 p# 4.014476939333037e+48
 31 p  131 p# 5.258964790526278e+50
 32 p  137 p# 7.204

In [66]:
smallp[-1]

np.int64(9699691)